<a href="https://colab.research.google.com/github/shahriarSwanon/Image-classification/blob/main/ImageClassiferKNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# No drive mounting needed
!git clone https://github.com/shahriarSwanon/Image-classification.git

# Verify the dataset structure
!ls "Image-classification/Animal Dataset"

Cloning into 'Image-classification'...
remote: Enumerating objects: 632, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 632 (delta 32), reused 30 (delta 3), pack-reused 544 (from 2)
Receiving objects: 100% (632/632), 612.19 MiB | 29.15 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Updating files: 100% (550/550), done.
cat  cow  dog  lamb  zebra


In [3]:
!pip install opencv-python-headless scikit-learn numpy matplotlib seaborn


In [4]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def load_images_from_path(base_path, img_size=(64, 64)):
    images = []
    labels = []
    class_names = []

    # Get class folders
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path):
            class_names.append(item)

    print(f"Found classes: {class_names}")

    # Load images
    for class_idx, class_name in enumerate(class_names):
        class_path = os.path.join(base_path, class_name)
        image_files = [f for f in os.listdir(class_path)
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Loading {len(image_files)} images from {class_name}")

        for img_file in image_files:
            img_path = os.path.join(class_path, img_file)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, img_size)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img)
                labels.append(class_idx)

    return np.array(images), np.array(labels), class_names

# Load dataset
base_path = "Image-classification/Animal Dataset"
X, y, class_names = load_images_from_path(base_path)

print(f"\nTotal images: {len(X)}")
print(f"Images per class: {len(X)//len(class_names)}")

Found classes: ['lamb', 'zebra', 'dog', 'cat', 'cow']
Loading 107 images from lamb
Loading 108 images from zebra
Loading 105 images from dog
Loading 111 images from cat
Loading 101 images from cow

Total images: 532
Images per class: 106


In [6]:
def extract_features(images):
    features = []
    for img in images:
        # Color histograms
        hist_features = []
        for channel in range(3):
            hist = cv2.calcHist([img], [channel], None, [32], [0, 256])
            hist = cv2.normalize(hist, hist).flatten()
            hist_features.extend(hist)

        # Edge features
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        edge_features = np.histogram(edges, bins=16)[0]

        # Combine
        all_features = np.concatenate([hist_features, edge_features])
        features.append(all_features)

    return np.array(features)

X_features = extract_features(X)
print(f"Feature vector size: {X_features.shape[1]}")

Feature vector size: 112


In [7]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42, stratify=y
)

# Normalize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Find best K
best_accuracy = 0
best_k = 1

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)

    if acc > best_accuracy:
        best_accuracy = acc
        best_k = k

    print(f"K={k:2d}, Accuracy={acc*100:.2f}%")

print(f"\n✅ Best K={best_k} with {best_accuracy*100:.2f}% accuracy")

# Train final model
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)
y_pred_final = final_knn.predict(X_test_scaled)
final_accuracy = accuracy_score(y_test, y_pred_final) * 100

print(f"\n{'='*50}")
print(f"🎯 FINAL ACCURACY: {final_accuracy:.2f}%")
print(f"{'='*50}")

K= 1, Accuracy=47.66%
K= 2, Accuracy=45.79%
K= 3, Accuracy=36.45%
K= 4, Accuracy=34.58%
K= 5, Accuracy=36.45%
K= 6, Accuracy=33.64%
K= 7, Accuracy=32.71%
K= 8, Accuracy=31.78%
K= 9, Accuracy=34.58%
K=10, Accuracy=32.71%
K=11, Accuracy=36.45%
K=12, Accuracy=34.58%
K=13, Accuracy=31.78%
K=14, Accuracy=34.58%
K=15, Accuracy=34.58%
K=16, Accuracy=36.45%
K=17, Accuracy=35.51%
K=18, Accuracy=35.51%
K=19, Accuracy=33.64%
K=20, Accuracy=35.51%

✅ Best K=1 with 47.66% accuracy

🎯 FINAL ACCURACY: 47.66%


In [8]:
# Save to Colab's temporary storage
import joblib
joblib.dump(final_knn, 'knn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Download to your computer
from google.colab import files
files.download('knn_model.pkl')
files.download('scaler.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>